### Importando as Bibliotecas

In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression

def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

def plot_profit_acu(dataframe, title_text):
    dataframe['Profit_acu'] = dataframe.Profit.cumsum()
    n_apostas = dataframe.shape[0]
    profit = round(dataframe.Profit_acu.tail(1).item(), 2)
    ROI = round((dataframe.Profit_acu.tail(1) / n_apostas * 100).item(), 2)
    drawdown = dataframe['Profit_acu'] - dataframe['Profit_acu'].cummax()
    drawdown_maximo = round(drawdown.min(), 2)
    winrate_medio = round((dataframe['Profit'] > 0).mean() * 100, 2)
    desvio_padrao = round(dataframe['Profit'].std(), 2)
    dataframe.Profit_acu.plot(title=title_text, xlabel='Entradas', ylabel='Stakes')
    print("Método:",title_text)
    print("Profit:", profit, "stakes em", n_apostas, "jogos")
    print("ROI:", ROI, "%")
    print("Drawdown Maximo Acumulado:", drawdown_maximo)
    print("Winrate Medio:", winrate_medio, "%")
    print("Desvio Padrao:", desvio_padrao)
    print("")

In [ ]:
ligas = [
# 'ARGENTINA - LIGA PROFESIONAL',
# 'ARGENTINA - COPA DE LA LIGA PROFESIONAL',
# 'ARMENIA - PREMIER LEAGUE',
# 'AUSTRALIA - A-LEAGUE',
# 'AUSTRIA - 2. LIGA',
# 'AUSTRIA - BUNDESLIGA',
# 'BELGIUM - CHALLENGER PRO LEAGUE',
# 'BELGIUM - JUPILER PRO LEAGUE',
# 'BOSNIA AND HERZEGOVINA - PREMIJER LIGA BIH',
# 'BRAZIL - COPA DO BRASIL',
# 'BRAZIL - SERIE A',
# 'BRAZIL - SERIE B',
# 'BULGARIA - PARVA LIGA',
# 'CHINA - SUPER LEAGUE',
# 'CROATIA - HNL',
# 'CROATIA - PRVA NL',
# 'CZECH REPUBLIC - FORTUNA:LIGA',
# 'DENMARK - 1ST DIVISION',
# 'DENMARK - SUPERLIGA',
# 'EGYPT - PREMIER LEAGUE',
# 'ENGLAND - CHAMPIONSHIP',
# 'ENGLAND - LEAGUE ONE',
# 'ENGLAND - LEAGUE TWO',
# 'ENGLAND - NATIONAL LEAGUE',
# 'ENGLAND - PREMIER LEAGUE',
# 'ESTONIA - ESILIIGA',
# 'ESTONIA - MEISTRILIIGA',
# 'EUROPE - CHAMPIONS LEAGUE',
# 'EUROPE - EUROPA CONFERENCE LEAGUE',
# 'EUROPE - EUROPA LEAGUE',
# 'FINLAND - VEIKKAUSLIIGA',
# 'FINLAND - YKKONEN',
# 'FRANCE - LIGUE 1',
# 'FRANCE - LIGUE 2',
# 'FRANCE - NATIONAL',
# 'GERMANY - 2. BUNDESLIGA',
# 'GERMANY - 3. LIGA',
# 'GERMANY - BUNDESLIGA',
# 'HUNGARY - OTP BANK LIGA',
# 'ICELAND - BESTA DEILD KARLA',
# 'IRELAND - PREMIER DIVISION',
# 'ITALY - SERIE A',
# 'ITALY - SERIE B',
# 'JAPAN - J1 LEAGUE',
# 'JAPAN - J2 LEAGUE',
# 'MEXICO - LIGA DE EXPANSION MX',
# 'MEXICO - LIGA MX',
# 'NETHERLANDS - EERSTE DIVISIE',
# 'NETHERLANDS - EREDIVISIE',
# 'NORWAY - ELITESERIEN',
# 'NORWAY - OBOS-LIGAEN',
# 'POLAND - EKSTRAKLASA',
# 'PORTUGAL - LIGA PORTUGAL',
# 'PORTUGAL - LIGA PORTUGAL 2',
# 'ROMANIA - LIGA 1',
# 'SAUDI ARABIA - SAUDI PROFESSIONAL LEAGUE',
# 'SCOTLAND - CHAMPIONSHIP',
# 'SCOTLAND - LEAGUE ONE',
# 'SCOTLAND - LEAGUE TWO',
# 'SCOTLAND - PREMIERSHIP',
# 'SERBIA - SUPER LIGA',
# 'SLOVAKIA - NIKE LIGA',
# 'SLOVENIA - PRVA LIGA',
# 'SOUTH AMERICA - COPA LIBERTADORES',
# 'SOUTH AMERICA - COPA SUDAMERICANA',
# 'SOUTH KOREA - K LEAGUE 1',
# 'SOUTH KOREA - K LEAGUE 2',
'SPAIN - LALIGA',
# 'SPAIN - LALIGA2',
# 'SWEDEN - ALLSVENSKAN',
# 'SWEDEN - SUPERETTAN',
# 'SWITZERLAND - CHALLENGE LEAGUE',
# 'SWITZERLAND - SUPER LEAGUE',
# 'TURKEY - 1. LIG',
# 'TURKEY - SUPER LIG',
# 'UKRAINE - PREMIER LEAGUE',
# 'USA - MLS',
# 'WALES - CYMRU PREMIER'
]

### Importando a Base de Dados e Criando Variáveis

In [ ]:
url = 'https://github.com/futpythontrader/YouTube/raw/main/Base_de_Dados/futpythontraderpunter.csv'
base = pd.read_csv(url)
base = base[['Date','League','Home','Away','Goals_H_FT','Goals_A_FT','Odd_H','Odd_D','Odd_A','Odd_Over25_FT','Odd_Under25_FT','Odd_BTTS_Yes','Odd_BTTS_No']]
base.columns = ['Date','League','Home','Away','Goals_H','Goals_A','Odd_H','Odd_D','Odd_A','Odd_Over25','Odd_Under25','Odd_BTTS_Yes','Odd_BTTS_No']
base = base[base['League'].isin(ligas) == True]
base['Date'] = pd.to_datetime(base['Date'])
base = base.sort_values(by='Date')
base = drop_reset_index(base)

In [ ]:
# Probabilidades
base['p_H'] = 1 / base['Odd_H']
base['p_D'] = 1 / base['Odd_D']
base['p_A'] = 1 / base['Odd_A']

# Razão das Odds
base['H_D'] = base['Odd_H'] / base['Odd_D']
base['H_A'] = base['Odd_H'] / base['Odd_A']

base = drop_reset_index(base)

### Inicializando o Modelo

In [ ]:
treino = base[base['Date'].dt.year <= 2022]
teste = base[base['Date'].dt.year >= 2023]

In [ ]:
treino

In [ ]:
teste

In [ ]:
features = ['p_H','p_D','p_A','H_D','H_A']

target = 'Odd_H'

In [ ]:
X_train = treino[features]
y_train = treino[target]

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
X_test = teste[features]

In [ ]:
Predicted_Odd = model.predict(X_test)

In [ ]:
teste['Predicted_Odd'] = Predicted_Odd

In [ ]:
teste.head()

### Criando um Modelo a Partir da Regressão

In [ ]:
# Modelo de Back Home
flt = (teste.Predicted_Odd > teste.Odd_H) & (teste.Odd_H < teste.Odd_A)
df = teste[flt]
df = drop_reset_index(df)

df['Profit'] = -1
df.loc[(df['Goals_H'] > df['Goals_A']), 'Profit'] = df['Odd_H'] - 1
plot_profit_acu(df, 'Back Home')

In [ ]:
# Modelo de Lay Away
flt = (teste.Predicted_Odd > teste.Odd_H) & (teste.Odd_H < teste.Odd_A)
df = teste[flt]
df = drop_reset_index(df)

df['Profit'] = -1
df.loc[(df['Goals_H'] >= df['Goals_A']), 'Profit'] = 1 / (df['Odd_A'] - 0.5)
plot_profit_acu(df, 'Lay Away')

In [ ]:
intercept = model.intercept_
coefficients = model.coef_

equation = f"Jogos_do_Dia['Predicted_Odd'] = {intercept}"

for coef, feature in zip(coefficients, features):
    equation += f" + ({coef}) * Jogos_do_Dia['{feature}']"

print(equation)

### Entradas

In [ ]:
dia = '2024-04-20'
url = f'https://github.com/futpythontrader/YouTube/raw/main/Jogos_do_Dia_FlashScore/{dia}_Jogos_do_Dia_FlashScore.csv'
Jogos_do_Dia = pd.read_csv(url)
Jogos_do_Dia = Jogos_do_Dia[Jogos_do_Dia['League'].isin(ligas) == True]
Jogos_do_Dia = Jogos_do_Dia[['Date','Time','League','Home','Away','Odd_H','Odd_D','Odd_A']]
Jogos_do_Dia.columns = ['Date','Time','League','Home','Away','Odd_H','Odd_D','Odd_A']
Jogos_do_Dia = drop_reset_index(Jogos_do_Dia)

# Probabilidades
Jogos_do_Dia['p_H'] = 1 / Jogos_do_Dia['Odd_H']
Jogos_do_Dia['p_D'] = 1 / Jogos_do_Dia['Odd_D']
Jogos_do_Dia['p_A'] = 1 / Jogos_do_Dia['Odd_A']

# Razão das Odds
Jogos_do_Dia['H_D'] = Jogos_do_Dia['Odd_H'] / Jogos_do_Dia['Odd_D']
Jogos_do_Dia['H_A'] = Jogos_do_Dia['Odd_H'] / Jogos_do_Dia['Odd_A']

# # Equação
Jogos_do_Dia['Predicted_Odd'] = 3.672202168106155 + (-2.5709596242967723) * Jogos_do_Dia['p_H'] + (-2.635638645908915) * Jogos_do_Dia['p_D'] + (-2.5683325952775737) * Jogos_do_Dia['p_A'] + (0.969174739426463) * Jogos_do_Dia['H_D'] + (0.9371683771139556) * Jogos_do_Dia['H_A']

flt = (Jogos_do_Dia.Predicted_Odd > Jogos_do_Dia.Odd_H) & (Jogos_do_Dia.Odd_H < Jogos_do_Dia.Odd_A)
Entradas = Jogos_do_Dia[flt]
Entradas = drop_reset_index(Entradas)
Entradas = Entradas[['Date','Time','League','Home','Away','Odd_H','Odd_D','Odd_A']]
Entradas

### Resultados

In [ ]:
flt = base.Date == str(dia)
base_today = base[flt]
base_today = base_today[['League','Home','Away','Goals_H','Goals_A']]
base_today = drop_reset_index(base_today)

Entradas = pd.merge(Entradas, base_today, on=['League','Home', 'Away'])
Entradas = drop_reset_index(Entradas)
Entradas